# 10 — From landscape to review (Book 2, Chapter 10)

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
from scipy import stats
ROOT = Path("..").resolve() if Path("../config.yml").exists() else Path(".").resolve()
RAW, FIN, LOGS = ROOT/"data/raw", ROOT/"data/final", ROOT/"logs"
pm = pd.read_json(sorted(RAW.glob("pubmed_*.jsonl"))[-1], lines=True, dtype={"pmid": str})[["pmid","title","year","pub_types"]]
ent = pd.read_parquet(FIN/"entities.parquet").drop(columns="year")
d = pm.merge(ent, on="pmid").reset_index(drop=True)

## 1. Randomised trials in the corpus (indexer-assigned publication type)

In [2]:
d["rct"] = d.pub_types.map(lambda p: "Randomized Controlled Trial" in p)
d["sr"] = d.pub_types.map(lambda p: "Systematic Review" in p or "Meta-Analysis" in p)
print("RCT reports:", d.rct.sum(), "| systematic reviews or meta-analyses:", d.sr.sum())
rct = d[d.rct]
print(rct.explode("candidates")["candidates"].value_counts().head(10).to_string())
rct[["pmid","year","title","candidates","phase","trial_ids"]].to_csv(LOGS/"rct_list_for_screening.csv", index=False)

RCT reports: 371 | systematic reviews or meta-analyses: 99
candidates
RTS,S      93
PfSPZ      37
AMA1       32
SPf66      25
MSP1       21
ME-TRAP    14
GMZ2       11
MSP3       10
FMP2.1      8
R21         8


## 2. Evidence gap map: target stage × type of evidence

In [3]:
def ev(row):
    pt = row.pub_types
    if "Clinical Trial, Phase III" in pt or "3" in row.phase: return "4 Phase 3"
    if "Clinical Trial, Phase II" in pt or "2" in row.phase: return "3 Phase 2"
    if "Clinical Trial, Phase I" in pt or "1" in row.phase: return "2 Phase 1"
    if "Systematic Review" in pt or "Meta-Analysis" in pt: return "5 Review"
    return "1 Other (lab, animal, observational, commentary)"
d["evidence"] = d.apply(ev, axis=1)
gm = d.reset_index(drop=True).explode("stage").dropna(subset=["stage"])
gap = gm.groupby(["stage", "evidence"]).size().unstack(fill_value=0); gap.to_csv(LOGS/"gap_map_stage_evidence.csv"); gap

evidence,"1 Other (lab, animal, observational, commentary)",2 Phase 1,3 Phase 2,4 Phase 3,5 Review
stage,,,,,
blood-stage,2781,131,73,27,9
pre-erythrocytic,2672,106,136,174,16
transmission-blocking,963,43,8,16,5


## 3. Worked meta-analysis (teaching example: two estimates from the same R21 phase 3 trial)
Vaccine efficacy (VE) = 1 − HR. Convert each 95% CI to a standard error on the log HR scale.

In [4]:
est = pd.DataFrame({"site": ["seasonal", "standard"], "ve": [75, 68], "lo": [71, 61], "hi": [79, 74]})
est["loghr"] = np.log(1 - est.ve/100)
est["se"] = (np.log(1 - est.lo/100) - np.log(1 - est.hi/100)) / (2*1.96)
w = 1/est.se**2; fe = (w*est.loghr).sum()/w.sum(); se_fe = np.sqrt(1/w.sum())
Q = (w*(est.loghr - fe)**2).sum(); dfree = len(est) - 1; I2 = max(0, (Q - dfree)/Q)
tau2 = max(0, (Q - dfree)/(w.sum() - (w**2).sum()/w.sum())); wr = 1/(est.se**2 + tau2)
re = (wr*est.loghr).sum()/wr.sum(); se_re = np.sqrt(1/wr.sum())
print(est.round(4).to_string(index=False))
print(f"Fixed effect: log HR {fe:.3f}, HR {np.exp(fe):.2f}, VE {100*(1-np.exp(fe)):.1f}% "
      f"(95% CI {100*(1-np.exp(fe+1.96*se_fe)):.1f}–{100*(1-np.exp(fe-1.96*se_fe)):.1f})")
print(f"Q = {Q:.2f}, df = {dfree}, p = {stats.chi2.sf(Q, dfree):.3f}, I² = {100*I2:.0f}%")
print(f"Random effects (DerSimonian–Laird): tau² = {tau2:.4f}, VE {100*(1-np.exp(re)):.1f}% "
      f"(95% CI {100*(1-np.exp(re+1.96*se_re)):.1f}–{100*(1-np.exp(re-1.96*se_re)):.1f})")

    site  ve  lo  hi   loghr     se
seasonal  75  71  79 -1.3863 0.0823
standard  68  61  74 -1.1394 0.1034
Fixed effect: log HR -1.291, HR 0.28, VE 72.5% (95% CI 68.8–75.8)
Q = 3.49, df = 1, p = 0.062, I² = 71%
Random effects (DerSimonian–Laird): tau² = 0.0217, VE 71.9% (95% CI 64.3–78.0)
